In [58]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [59]:
class MultipleOLS:
    
    def __init__(self):
        self.weights = None
        self.intercept = None
        
    def fit(self, X , y):
        X = np.asarray(X)
        y = np.asarray(y).reshape(-1,1)
        
        X = np.insert(X , 0 , 1 , axis = 1)
        weight_and_intercept = np.linalg.inv(X.T @ X) @ X.T @ y
        self.intercept = weight_and_intercept[0,0]
        self.weights = weight_and_intercept[1: , 0]
        return self.weights , self.intercept
    
    def predict(self , X):
        X = np.asarray(X)
        return X@self.weights + self.intercept
        

In [60]:
df = pd.read_csv("../../Data/housing.csv")

In [61]:
df.head(5)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [63]:
df.isnull().mean()*100

longitude             0.000000
latitude              0.000000
housing_median_age    0.000000
total_rooms           0.000000
total_bedrooms        1.002907
population            0.000000
households            0.000000
median_income         0.000000
median_house_value    0.000000
ocean_proximity       0.000000
dtype: float64

In [64]:
df['ocean_proximity'].unique()

array(['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND'],
      dtype=object)

In [65]:
df = df.dropna(subset=['total_bedrooms'])

In [66]:
from sklearn.model_selection import train_test_split
X_train ,X_test , y_train ,y_test = train_test_split(df.drop(columns=['median_house_value']) , df['median_house_value'] , test_size=0.2)

In [67]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

trf = ColumnTransformer([
    ("category encode" , OneHotEncoder(drop = 'first' , handle_unknown="ignore") , ['ocean_proximity'])
], remainder= "passthrough")

X_train_processed = trf.fit_transform(X_train)
X_test_processed = trf.transform(X_test)

In [68]:
from sklearn.metrics import r2_score
ols = MultipleOLS()
ols.fit(X_train_processed, y_train)
y_pred = ols.predict(X_test_processed)
print(r2_score(y_test , y_pred))

0.6599124198936995


## Sklearn implementation

In [71]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train_processed , y_train)
y_pred = lr.predict(X_test_processed)
print(r2_score(y_test , y_pred))

0.659912419893222
